In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import Tensor
from jaxtyping import Float, Int

In [ ]:
class RNNModel(nn.Module):
    def __init__(
        self, vocab_size: int = 65, n_embed: int = 32, hidden_size: int = 64
    ) -> None:
        super().__init__()
        self.hidden_size = hidden_size

        self.table = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=n_embed
        )  # (65, 32)
        self.linear1 = nn.Linear(
            n_embed + self.hidden_size, hidden_size
        )  # (96{32 + 64}, 64) -> (bs, 64)
        self.tan_h = nn.Tanh()
        # Create linear2: projects from hidden space to vocab_size to produce logits (one score per token)
        self.linear2 = nn.Linear(hidden_size, vocab_size)  # (64, 65)

    # idx is tensor (B, T), batch and timestampt/sequence
    def forward(self, idx: Tensor, targets=None):
        B, T = idx.shape
        h0 = torch.zeros(
            [B, self.hidden_size], device=idx.device
        )  # (B, hidden_size) hidden memory, match to idx device to prevent mismatch

        # Embed all tokens at once using the embedding table
        embed = self.table(idx)  # (B,T,n_embed) each id becomes a vector
        # Create an empty list to collect the logit output at each timestep
        logits = []
        # Loop over each timestep t from 0 to T-1
        for t in range(T):
            # embedding for timestep t
            # B token embeddings, one per sequence at position t
            x_t = embed[:, t, :]  # B,n_embed

            combined = torch.cat([x_t, h0], dim=-1)  # (B, num_embed + hidden)

            out1 = self.linear1(combined)
            h0 = self.tan_h(out1)
            out2 = self.linear2(h0)  # (B, vocab_size), predictions
            logits.append(out2)

        # Stack the list of T logit tensors into a single tensor along dimension 1
        # Goes from a list of T tensors of shape (B, vocab_size) → (B, T, vocab_size)
        # Now we have one prediction per timestep, for every item in the batch
        stacked = torch.stack(logits, dim=1)
        # Initialize loss as None (only computed if targets are provided)
        loss = None
        # If targets were passed in, compute the cross entropy loss
        if targets is not None:
            # Unpack logits shape into B, T, C (C = vocab_size)
            B, T, C = stacked.shape
            # Reshape logits from (B, T, C) → (B*T, C) — cross_entropy expects (N, C)
            reshaped1 = torch.reshape(stacked, (B * T, C))
            # Reshape targets from (B, T) → (B*T,) — cross_entropy expects (N,) for class indices
            targets = torch.reshape(targets, (B * T,))
            # Call F.cross_entropy on the reshaped logits and targets
            loss = F.cross_entropy(reshaped1, targets)
        # Return logits and loss
        return stacked, loss


In [4]:
emb = nn.Embedding(12, 6)

In [5]:
emb.weight

Parameter containing:
tensor([[-0.0441, -0.2194,  0.0772,  0.6665,  1.6316, -0.1757],
        [ 0.5406,  0.6253, -0.0087, -0.9071,  0.7998, -0.8933],
        [ 1.4082,  0.5872,  0.1358,  2.0664,  1.5407,  0.7161],
        [ 1.2742, -0.3001,  0.4790,  0.5727,  0.5908, -0.9521],
        [-1.0012,  0.7780, -1.2771, -0.4007, -2.4586, -0.3184],
        [ 0.6148, -0.1350, -0.4583, -1.2099,  0.1263, -0.4550],
        [ 0.2628,  0.7409, -0.7733,  1.3663,  0.3684, -0.5915],
        [-0.4785,  0.7812, -0.2433, -0.0035, -1.1816,  0.8248],
        [ 0.2738, -0.1702, -1.1601,  0.8645,  1.0324, -1.1223],
        [ 0.6820, -0.5910,  2.5214,  0.0485,  0.3566,  0.0874],
        [-0.3787, -1.8228, -0.0163, -3.0440, -1.9795,  0.6366],
        [ 1.0650, -1.7079,  0.5904,  1.0684, -0.4109,  0.2956]],
       requires_grad=True)

In [6]:
emb(torch.tensor([[1, 2, 3], [1, 2, 3]]))

tensor([[[ 0.5406,  0.6253, -0.0087, -0.9071,  0.7998, -0.8933],
         [ 1.4082,  0.5872,  0.1358,  2.0664,  1.5407,  0.7161],
         [ 1.2742, -0.3001,  0.4790,  0.5727,  0.5908, -0.9521]],

        [[ 0.5406,  0.6253, -0.0087, -0.9071,  0.7998, -0.8933],
         [ 1.4082,  0.5872,  0.1358,  2.0664,  1.5407,  0.7161],
         [ 1.2742, -0.3001,  0.4790,  0.5727,  0.5908, -0.9521]]],
       grad_fn=<EmbeddingBackward0>)

In [7]:
vocab_size: int = 65
n_embed: int = 32
hidden_size: int = 64

In [8]:
batch = torch.randint(0, 5, (4, 8))

In [9]:
batch

tensor([[0, 1, 0, 2, 3, 2, 2, 0],
        [4, 3, 1, 0, 0, 4, 2, 2],
        [0, 0, 0, 0, 4, 0, 3, 4],
        [0, 4, 4, 2, 1, 3, 4, 3]])

In [10]:
B, T = batch.shape
B, T

(4, 8)

In [11]:
table = nn.Embedding(num_embeddings=65, embedding_dim=32)  # (65, 32)

In [12]:
embed = table(batch)
embed.shape

torch.Size([4, 8, 32])

In [13]:
# batch at position 0
x_t = embed[:, 0, :]

In [14]:
# B(4) token embeddings at position 0
x_t.shape

torch.Size([4, 32])

In [15]:
h0 = torch.zeros([4, 64])  # B, hidden_size
h0.shape

torch.Size([4, 64])

In [16]:
x_t.shape, h0.shape

(torch.Size([4, 32]), torch.Size([4, 64]))

In [17]:
combined = torch.cat([x_t, h0], dim=-1)
combined.shape

torch.Size([4, 96])

In [18]:
linear1 = nn.Linear(32 + 64, 64)  # (96{hidden_size + n_embed}, hidden_size) -> ()

In [19]:
linear1(combined).shape

torch.Size([4, 64])

In [21]:
lin1_out = linear1(combined)
lin1_out.max()

tensor(0.7867, grad_fn=<MaxBackward1>)

In [22]:
tan_h = nn.Tanh()
tanlin = tan_h(lin1_out)
tanlin.shape
tanlin.max()

tensor(0.6565, grad_fn=<MaxBackward1>)

In [23]:
lin2 = nn.Linear(hidden_size, vocab_size)
out = lin2(tanlin)

In [25]:
out.shape

torch.Size([4, 65])

In [ ]:
logits = []
for i in range(4):
    logits.append(out)

In [40]:
stacked = torch.stack(logits, dim=1)

In [43]:
B, T, C = stacked.shape

In [ ]:
torch.reshape(stacked, (B * T, C)).shape

torch.Size([16, 65])